# 🎬 Chapitrage vidéo des Conseils communaux — PV Explorer

Extrait, depuis la chaîne YouTube **@1030be**, le chapitrage (points débattus,
auteur·e, deep-link) des séances filmées du Conseil communal de Schaerbeek.

Ce notebook télécharge et exécute **toujours la dernière version** du script
[`pipeline/extract_video_chapters.py`](https://github.com/pmeyssonnier/pv-explorer-app/blob/main/pipeline/extract_video_chapters.py)
du dépôt — rien n'est dupliqué ici, donc rien ne devient obsolète.

Deux usages, indépendants :
- **① → ③ Extraction complète** : scanne toute la chaîne (`/videos` + `/streams`)
  et régénère les deux fichiers en entier. À utiliser périodiquement (nouvelles
  séances) ou après une amélioration du script.
- **④ Ajout manuel (optionnel)** : ajoute/actualise **une seule** séance que tu
  as repérée toi-même sur YouTube (titre non standard, playlist… — hors du
  scan automatique), sans relancer tout le reste.


## ① Dépendances

In [ ]:
!pip install -U yt-dlp -q
!curl -fsSL https://deno.land/install.sh | DENO_INSTALL=/usr/local sh


## ② Charger le script (fonctions à jour du dépôt)

Télécharge et **définit** les fonctions du script (sans encore rien exécuter :
`__name__` est neutralisé pour que le scan complet ne se lance pas tout seul).

In [ ]:
import json
import types
import urllib.request

url = "https://raw.githubusercontent.com/pmeyssonnier/pv-explorer-app/main/pipeline/extract_video_chapters.py"
code_src = urllib.request.urlopen(url).read().decode("utf-8")

_mod = types.ModuleType("extract_video_chapters")
_mod.__dict__["__name__"] = "extract_video_chapters"   # neutralise `if __name__ == "__main__":`
exec(compile(code_src, "extract_video_chapters.py", "exec"), _mod.__dict__)
globals().update(_mod.__dict__)
print("Fonctions chargées :", "main, build_seance_entry, merge_seance, ...")


## ③ Extraction complète (toutes les séances du canal) — optionnel

In [ ]:
main()  # noqa: F821 -- chargé dynamiquement par la cellule ② (exec + globals().update())
# Le script lance le scan complet et écrit deux fichiers dans /content/ :
#  - pv_video_conseil_schaerbeek.json (chapitres + auteur·e·s + deep-links)
#  - video_sessions.json (date → URL de la vidéo de séance)


### Et ensuite ?

Télécharge les deux fichiers produits (panneau fichiers à gauche, ⋮ → *Download*)
et transmets-les pour intégration — **aucune réindexation Pinecone n'est
nécessaire**, ces fichiers sont lus directement par le backend.


## ④ Ajouter une séance repérée manuellement (optionnel)

Pour une vidéo que tu as trouvée toi-même sur YouTube (hors scan automatique :
titre non standard, playlist…), sans relancer tout le scan. Nécessite d'avoir
exécuté **① et ②** (pas besoin de ③).

Renseigne l'URL ci-dessous — la date est déduite du titre de la vidéo si
possible (« Conseil communal du JJ/MM/AAAA »), sinon précise-la explicitement
avec `date="AAAA-MM-JJ"`.

In [ ]:
VIDEO_URL = "https://www.youtube.com/watch?v=XXXXXXXXXXX"  # ← à remplacer
# DATE = "2026-07-15"   # décommente si la date n'est pas dans le titre

seance = build_seance_entry(VIDEO_URL)  # noqa: F821 -- idem, chargé par la cellule ②

# Fusionne avec les fichiers ACTUELS du dépôt (upsert par video_id — relancer
# sur la même vidéo la met simplement à jour, sans doublon).
base = "https://raw.githubusercontent.com/pmeyssonnier/pv-explorer-app/main/backend/"
chapitres = json.loads(urllib.request.urlopen(base + "video_conseil_schaerbeek.json").read())
sessions = json.loads(urllib.request.urlopen(base + "video_sessions.json").read())

chapitres["seances"] = merge_seance(chapitres["seances"], seance)  # noqa: F821
sessions[seance["date"]] = seance["video_url"]
sessions = dict(sorted(sessions.items(), reverse=True))

with open("/content/video_conseil_schaerbeek.json", "w", encoding="utf-8") as f:
    json.dump(chapitres, f, ensure_ascii=False, indent=2)
with open("/content/video_sessions.json", "w", encoding="utf-8") as f:
    json.dump(sessions, f, ensure_ascii=False, indent=2)

n_seances = len(chapitres["seances"])
print(f"✅ {n_seances} séances (dont celle ajoutée) → /content/video_conseil_schaerbeek.json")
print(f"✅ {len(sessions)} séances filmées → /content/video_sessions.json")


### Et ensuite ?

Télécharge les deux fichiers `/content/video_conseil_schaerbeek.json` et
`/content/video_sessions.json` et transmets-les pour intégration (remplacement
direct des fichiers du même nom dans `backend/`) — toujours **aucune
réindexation Pinecone requise** pour ces deux fichiers.
